In [ ]:
%pip -q install openai pandas numpy scipy scikit-learn tqdm

In [ ]:
from google.colab import drive, userdata
from IPython.display import display
from pathlib import Path
import base64
import json
import re
import time

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from openai import OpenAI

RUN_TRAIN_API_CALLS = False
RUN_DEV_API_CALLS = False
RUN_TEST_API_CALLS = False
CI_MODEL = 'gpt-5.5'
CI_REASONING_EFFORT = 'none'
CI_IMAGE_DETAIL = 'original'
REQUEST_SLEEP_SECONDS = 0.2
MAX_RETRIES = 3

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Dr. Lulwah - Ahmed/ImageEVAl')
PROJECT_DIR = ROOT / 'ImageEval2026_Task2_CRAI_Bench'
DATA_DIR = PROJECT_DIR / 'data'
EXPERIMENT_ROOT = PROJECT_DIR / 'ci_integrity_structured_v1'
CACHE_DIR = EXPERIMENT_ROOT / 'cache'
OUTPUT_DIR = EXPERIMENT_ROOT / 'outputs'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
client = OpenAI(api_key=userdata.get('openai'))

In [ ]:
DIM_COLS = ['CRAI_CEA', 'CRAI_CC', 'CRAI_CS', 'CRAI_CI', 'CRAI_HP']

def find_image(folder, stem):
    for extension in ['.png', '.jpg', '.jpeg', '.webp']:
        path = folder / f'{stem}{extension}'
        if path.exists():
            return path

def load_split(split, gold=False):
    folder = DATA_DIR / split
    frame = pd.read_csv(folder / 'captions.tsv', sep='\t')
    if gold:
        labels = pd.read_csv(folder / 'gold_human.tsv', sep='\t')
        keep = ['id'] + [x for x in DIM_COLS + ['CRAI_composite', 'category'] if x in labels]
        frame = frame.merge(labels[keep], on='id')
    frame['id'] = frame['id'].astype(str)
    frame['base_id'] = frame['id'].str.replace(r'_v\d+$', '', regex=True)
    frame['caption_version'] = frame['id'].str.extract(r'_v(\d+)$')[0].astype(int)
    frame['caption_version_key'] = 'v' + frame['caption_version'].astype(str)
    return frame.sort_values(['base_id', 'caption_version']).reset_index(drop=True)

train_df = load_split('train', gold=True)
dev_df = load_split('dev', gold=True)
test_df = load_split('test') if (DATA_DIR / 'test' / 'captions.tsv').exists() else pd.DataFrame()

In [ ]:
PROMPT_VERSION = 'ci-cultural-integrity-structured-v1'

CI_SYSTEM_PROMPT = r'''
ROLE

You are a strict multimodal evaluator for ImageEval 2026 CRAI-Bench. Judge only
CRAI_CI: Cultural Integrity for the Qatari cultural scene represented by the inputs.

AUTHOR DEFINITION

Cultural Integrity is the respectful and undistorted preservation of cultural
meaning.

INPUT AUTHORITY

- The REFERENCE IMAGE represents the intended cultural subject, practice, object,
  garment, place, or landmark.
- The V1 CAPTION names and explains culturally explicit identity-bearing details.
- The CURRENT CAPTION is the prompt used to generate the candidate. It may be more
  generic, but its loss of detail does not erase cultural meaning established by the
  reference and V1 caption.
- The GENERATED IMAGE is the candidate being scored.

DECISION PROCEDURE

1. Identify the smallest set of core features or relationships that carry the
   intended cultural meaning in the reference and V1 caption.
2. Decide whether the generated image still depicts that cultural subject or
   practice, rather than only satisfying a generic paraphrase.
3. Judge whether the identity-bearing form, use, handling, clothing construction,
   landmark geometry, or practice has been materially distorted.
4. Judge whether the representation is culturally plausible and respectful.
5. Assign the holistic CI score from the anchors below. Do not mechanically average
   the diagnostic fields.

SCORE ANCHORS

- 1.00: The intended cultural meaning is clearly recognizable, plausible,
  respectful, and materially undistorted. Composition and incidental detail may differ.
- 0.75: The core meaning is preserved. Noticeable changes or minor genericization
  exist, but identity and cultural function remain intact.
- 0.50: Cultural meaning is only partly preserved. A generic substitute, altered
  identity-bearing form, or important distortion weakens the intended meaning, but a
  meaningful portion remains recognizable.
- 0.25: Only a weak fragment or broad theme survives. Major genericization or
  distortion has removed most of the intended cultural meaning.
- 0.00: The intended cultural subject or practice is absent, replaced,
  unrecognizable, or materially corrupted, even if the generic current caption is met.

BOUNDARIES WITH OTHER CRAI METRICS

- Do not score CEA. Missing objects, counts, colors, or attributes affect CI only
  when their loss materially changes cultural identity or meaning.
- Do not score CC. Spatial or relational errors affect CI only when they corrupt the
  meaning of the practice, object, or landmark.
- Do not score CS. A scene may be less Qatar-specific yet retain integrity if its
  intended cultural meaning remains recognizable and undistorted.
- Do not score HP. Unsupported additions matter only when they distort meaning or
  make the representation disrespectful.
- Do not score general beauty, realism, lighting, or photographic quality unless a
  defect materially damages the cultural representation.

CRITICAL RULES

- A respectful generic scene is not enough when the intended identity is lost.
- Superficial resemblance or shared function is not enough for a named landmark,
  practice, garment, or cultural object.
- Do not require pixel-level similarity, identical camera angle, or every reference
  detail.
- Do not assume that traditional-looking, Arab-looking, Gulf-looking, or desert
  imagery automatically preserves the intended Qatari cultural meaning.
- Keep brief_reason to one sentence and return JSON only.

OUTPUT JSON

{
  "core_identity_retention": 0.0,
  "practice_or_form_integrity": 0.0,
  "respect_and_plausibility": 0.0,
  "generic_substitution_severity": 0.0,
  "material_distortion_severity": 0.0,
  "integrity_failure": "none | minor | substantial | total",
  "raw_ci": 0.0,
  "confidence": 0.0,
  "brief_reason": "one short sentence"
}
'''.strip()

In [ ]:
FAILURE_LEVELS = ['none', 'minor', 'substantial', 'total']
REQUIRED_NUMERIC = [
    'core_identity_retention', 'practice_or_form_integrity',
    'respect_and_plausibility', 'generic_substitution_severity',
    'material_distortion_severity', 'raw_ci', 'confidence',
]

def parse_json_object(text):
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip())
    return json.loads(text[text.find('{'):text.rfind('}') + 1])

def load_jsonl(path):
    if not path.exists():
        return []
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

def append_jsonl(path, record):
    with path.open('a') as handle:
        handle.write(json.dumps(record) + '\n')

def image_to_data_url(path):
    path = Path(path)
    mime = {'.png': 'image/png', '.jpg': 'image/jpeg',
            '.jpeg': 'image/jpeg', '.webp': 'image/webp'}[path.suffix.lower()]
    encoded = base64.b64encode(path.read_bytes()).decode('utf-8')
    return f'data:{mime};base64,{encoded}'

def split_image_paths(row):
    folder = DATA_DIR / row['split'] / 'imgs'
    return find_image(folder / 'ref', row['base_id']), find_image(folder / 'generated', row['id'])

def v1_caption_map(frame):
    v1 = frame[frame['caption_version'].eq(1)]
    return dict(zip(v1['base_id'], v1['caption']))

def call_ci_judge(row, v1_caption):
    reference, generated = split_image_paths(row)
    text = f"V1 caption: {v1_caption}\nCurrent caption: {row['caption']}"
    content = [
        {'type': 'input_text', 'text': text},
        {'type': 'input_text', 'text': 'REFERENCE IMAGE:'},
        {'type': 'input_image', 'image_url': image_to_data_url(reference), 'detail': CI_IMAGE_DETAIL},
        {'type': 'input_text', 'text': 'GENERATED IMAGE:'},
        {'type': 'input_image', 'image_url': image_to_data_url(generated), 'detail': CI_IMAGE_DETAIL},
    ]
    for attempt in range(MAX_RETRIES):
        try:
            response = client.responses.create(
                model=CI_MODEL,
                reasoning={'effort': CI_REASONING_EFFORT},
                input=[
                    {'role': 'developer', 'content': CI_SYSTEM_PROMPT},
                    {'role': 'user', 'content': content},
                ],
            )
            return parse_json_object(response.output_text)
        except Exception:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(2 ** (attempt + 1))

def load_or_infer(frame, split, run_calls):
    path = CACHE_DIR / f'ci_structured_{split}.jsonl'
    cache = {record['instance_id']: record for record in load_jsonl(path)}
    captions = v1_caption_map(frame)
    for _, row in tqdm(frame.iterrows(), total=len(frame), desc=f'CI {split}'):
        if row['id'] not in cache and run_calls:
            record = {
                'instance_id': row['id'],
                'response': call_ci_judge(row, captions[row['base_id']]),
            }
            cache[row['id']] = record
            append_jsonl(path, record)
            time.sleep(REQUEST_SLEEP_SECONDS)
    return pd.DataFrame([cache[x] for x in frame['id'] if x in cache])

train_records = load_or_infer(train_df.assign(split='train'), 'train', RUN_TRAIN_API_CALLS)
dev_records = load_or_infer(dev_df.assign(split='dev'), 'dev', RUN_DEV_API_CALLS)
test_records = load_or_infer(test_df.assign(split='test'), 'test', RUN_TEST_API_CALLS) if len(test_df) else pd.DataFrame()

In [ ]:
NUMERIC_FEATURES = REQUIRED_NUMERIC.copy()

def records_to_features(records, metadata):
    rows = []
    for record in records.to_dict('records'):
        response = record['response']
        row = {'id': record['instance_id']}
        row.update({name: float(response[name]) for name in REQUIRED_NUMERIC})
        row['integrity_failure'] = response['integrity_failure']
        rows.append(row)
    columns = ['id', 'base_id', 'caption_version', 'caption_version_key']
    return metadata[columns].merge(pd.DataFrame(rows), on='id') if rows else pd.DataFrame()

train_features = records_to_features(train_records, train_df)
dev_features = records_to_features(dev_records, dev_df)
test_features = records_to_features(test_records, test_df) if len(test_df) else pd.DataFrame()

In [ ]:
def safe_spearman(y, prediction):
    return float(spearmanr(np.asarray(y, float), np.asarray(prediction, float)).statistic)

def metrics(y, prediction):
    return {
        'spearman': safe_spearman(y, prediction),
        'mae': float(mean_absolute_error(y, prediction)),
    }

In [ ]:
NUMERIC_FEATURES = REQUIRED_NUMERIC.copy()
RIDGE_ALPHAS = [0.1, 1.0, 10.0, 100.0]

def design_matrix(frame):
    columns = [frame[name].to_numpy(float) for name in NUMERIC_FEATURES]
    columns.extend(frame['integrity_failure'].eq(level).to_numpy(float) for level in FAILURE_LEVELS)
    columns.extend(frame['caption_version'].eq(version).to_numpy(float) for version in range(1, 6))
    return np.column_stack(columns)

def select_alpha(training):
    groups = training['base_id'].to_numpy()
    splits = list(GroupKFold(5).split(training, groups=groups))
    rows = []
    predictions = {}
    for alpha in RIDGE_ALPHAS:
        oof = np.zeros(len(training))
        for train_index, valid_index in splits:
            scaler = StandardScaler().fit(design_matrix(training.iloc[train_index]))
            model = Ridge(alpha=alpha).fit(
                scaler.transform(design_matrix(training.iloc[train_index])),
                training.iloc[train_index]['CRAI_CI'],
            )
            oof[valid_index] = model.predict(
                scaler.transform(design_matrix(training.iloc[valid_index]))
            )
        predictions[alpha] = np.clip(oof, 0, 1)
        rows.append({
            'alpha': alpha,
            'spearman': safe_spearman(training['CRAI_CI'], predictions[alpha]),
            'mae': mean_absolute_error(training['CRAI_CI'], predictions[alpha]),
        })
    table = pd.DataFrame(rows).sort_values(['spearman', 'mae'], ascending=[False, True])
    alpha = float(table.iloc[0]['alpha'])
    return alpha, predictions[alpha]

def fit_model(training):
    alpha, oof = select_alpha(training)
    scaler = StandardScaler().fit(design_matrix(training))
    model = Ridge(alpha=alpha).fit(
        scaler.transform(design_matrix(training)), training['CRAI_CI']
    )
    calibrator = IsotonicRegression(out_of_bounds='clip').fit(oof, training['CRAI_CI'])
    return scaler, model, calibrator

def predict(model_parts, frame):
    scaler, model, calibrator = model_parts
    ridge_prediction = np.clip(model.predict(scaler.transform(design_matrix(frame))), 0, 1)
    return np.clip(calibrator.predict(ridge_prediction), 0, 1)

if len(train_features) == len(train_df):
    training = train_features.merge(train_df[['id', 'CRAI_CI']], on='id')
    fitted = fit_model(training)

    if len(dev_features) == len(dev_df):
        dev_output = pd.DataFrame({'id': dev_df['id'], 'prediction': predict(fitted, dev_features)})
        evaluation = dev_df[['id', 'CRAI_CI']].merge(dev_output, on='id')
        display(pd.DataFrame([metrics(evaluation['CRAI_CI'], evaluation['prediction'])]).round(4))
        evaluation.to_csv(OUTPUT_DIR / 'ci_dev_predictions.tsv', sep='\t', index=False)

    if len(test_features) == len(test_df) and len(test_df):
        test_output = pd.DataFrame({
            'id': test_df['id'],
            'CRAI_CI': predict(fitted, test_features),
        })
        test_output.to_csv(OUTPUT_DIR / 'ci_test_predictions.tsv', sep='\t', index=False)